In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
pip install ipywidgets

In [ ]:
!pip install evaluate rouge_score -q

In [1]:
import os, ast, re, warnings
import pandas as pd
import torch

warnings.filterwarnings("ignore")

from transformers import T5Tokenizer, T5ForConditionalGeneration

CSV_PATH   = "/content/drive/MyDrive/spells_master.csv"
MODEL_NAME = "t5-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head(3)

(1333, 20)


,name,desc,higher_levels,level,school,classes,subclasses,cast_time,range,duration,verbal,somatic,material,material_desc,concentration,ritual,damage_type,dc_type,attack_type,source
0,Prismatic Wall,"A shimmering, multicolored plane of light form...",NaN,9,Abjuration,['Wizard'],[],1 action,60 feet,10 minutes,True,True,False,NaN,False,False,NaN,NaN,NaN,wotc-srd
1,Symbol,"When you cast this spell, you inscribe a harmf...",NaN,7,Abjuration,"['Bard', 'Cleric', 'Wizard']",[],1 minute,Touch,Until dispelled or triggered,True,True,True,"Mercury, phosphorus, and powdered diamond and ...",False,False,NaN,NaN,NaN,wotc-srd
2,Teleport,This spell instantly transports you and up to ...,NaN,7,Conjuration,"['Bard', 'Sorcerer', 'Wizard']",[],1 action,10 feet,Instantaneous,True,False,False,NaN,False,False,NaN,NaN,NaN,wotc-srd


In [3]:
def parse_list_col(x):
    if pd.isna(x) or x == "": return []
    if isinstance(x, list):   return x
    try:    return ast.literal_eval(x)
    except: return [s.strip() for s in str(x).split(",") if s.strip()]

def clean(s):
    if not isinstance(s, str): return ""
    s = re.sub(r"\{@\w+\s([^}]+)\}",        r"\1",              s)
    s = re.sub(r"\{@\w+\}",                  "",                 s)
    s = re.sub(r"\|[A-Z]{2,}[^|\s]*",        "",                 s)
    s = re.sub(r"\[&\d+;&\d+\]",             "",                 s)
    s = re.sub(r"\|bestiary[^*\n]*",          "",                 s)
    s = re.sub(r"\|[^|}\s]{1,30}",           "",                 s)
    s = re.sub(r"Vision and Light\|[^\s]*",  "heavily obscured", s)
    s = re.sub(r"difficult terrain\|[^\s]*", "difficult terrain", s)
    return re.sub(r"\s+", " ", s).strip()

print("helpers ready")

helpers ready


In [4]:
torch.cuda.empty_cache()
import gc
gc.collect()

222

In [5]:
import torch
torch.cuda.empty_cache()

print(round(torch.cuda.memory_allocated() / 1e9, 2), 'GB used')
print(round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB total')

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
# bfloat16: native on L4/A100, same exponent range as float32 → no NaN risk
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
model = model.to(device)

print('model loaded')
print(round(torch.cuda.memory_allocated() / 1e9, 2), 'GB used after load')


0.0 GB used
42.41 GB total


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

model loaded
0.58 GB used after load


In [6]:
prompt = "describe spell: Fireball"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=40
    )

result = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(result)

spell: Fireball. Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball


In [7]:
STR_COLS  = ["desc", "higher_levels", "material_desc", "damage_type",
             "dc_type", "attack_type", "school", "cast_time", "range",
             "duration", "source"]
BOOL_COLS = ["verbal", "somatic", "material", "concentration", "ritual"]

for col in STR_COLS:
    if col in df.columns:
        df[col] = df[col].fillna("").apply(clean)

for col in BOOL_COLS:
    if col in df.columns:
        df[col] = df[col].apply(lambda x:
            bool(x) if isinstance(x, bool)
            else str(x).strip().lower() in ("true", "yes", "1", "t"))

for col in ["classes", "subclasses"]:
    if col in df.columns:
        df[col] = df[col].apply(parse_list_col)

df["name"]  = df["name"].fillna("").apply(clean)
df["level"] = pd.to_numeric(df["level"], errors="coerce").fillna(0).astype(int)

df = df[df["name"].str.strip() != ""].reset_index(drop=True)
df = df[df["desc"].str.len()   >  40].reset_index(drop=True)
df["cast_time"] = df["cast_time"].apply(lambda x: x if len(x) < 40 else "")

print(f"rows after cleaning : {len(df)}")
print(f"schools : {sorted(df['school'].unique().tolist())}")

rows after cleaning : 1333
schools : ['Abjuration', 'Conjuration', 'Divination', 'Enchantment', 'Evocation', 'Illusion', 'Necromancy', 'Transmutation']


In [8]:
class SpellDataset(torch.utils.data.Dataset):
    """Tokenizes all examples once at construction time.
    __getitem__ does zero CPU work — tensors are ready to go straight to GPU."""

    def __init__(self, data):
        inputs = tokenizer(
            [item["input"]  for item in data],
            max_length=128,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        targets = tokenizer(
            [item["target"] for item in data],
            max_length=256,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        label_ids = targets["input_ids"].clone()
        label_ids[label_ids == tokenizer.pad_token_id] = -100  # ignore padding in loss

        self.input_ids      = inputs["input_ids"]
        self.attention_mask = inputs["attention_mask"]
        self.labels         = label_ids

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }


In [ ]:
import numpy as np

# ── spell-type classifier ──────────────────────────────────────────────────
def infer_type(row):
    desc = (row.get("desc") or "").lower()
    rng  = (row.get("range") or "").lower().strip()
    dur  = (row.get("duration") or "").lower()

    has_aoe    = any(p in desc for p in ["-foot radius", "radius sphere", " cone", " cube", " cylinder", "line of "])
    is_instant = "instantaneous" in dur
    is_self    = rng in ("self", "")
    is_touch   = rng == "touch"
    has_damage = bool(str(row.get("damage_type", "")).strip())
    is_conc    = bool(row.get("concentration", False))

    if has_aoe:                                   return "aoe"
    if is_self and is_instant and not has_damage: return "teleport"
    if is_touch and not has_damage:               return "touch"
    if is_conc  and not has_damage:               return "concentration"
    if has_damage and not has_aoe:                return "single_target"
    if bool(row.get("ritual", False)):            return "ritual"
    return "utility"

# ── attribute string builder ───────────────────────────────────────────────
def build_attr(row):
    parts = [f"level: {row['level']}", f"type: {infer_type(row)}"]
    if row["school"]:        parts.append(f"school: {row['school']}")
    if row["cast_time"]:     parts.append(f"cast_time: {row['cast_time']}")
    if row["range"]:         parts.append(f"range: {row['range']}")
    if row["duration"]:      parts.append(f"duration: {row['duration']}")
    comps = [c for c, f in [("V", row["verbal"]), ("S", row["somatic"]), ("M", row["material"])] if f]
    if comps:                parts.append(f"components: {', '.join(comps)}")
    if row["classes"]:       parts.append(f"classes: {', '.join(row['classes'])}")
    if row["damage_type"]:   parts.append(f"damage: {row['damage_type']}")
    if row["dc_type"]:       parts.append(f"saving_throw: {row['dc_type']}")
    if row["concentration"]: parts.append("concentration: yes")
    if row["ritual"]:        parts.append("ritual: yes")
    if row.get("material_desc", ""): parts.append(f"material: {row['material_desc']}")
    return " | ".join(parts)

# ── text augmentation ──────────────────────────────────────────────────────
SWAPS = [
    (r"\bstreaks toward\b",             "flies at"),
    (r"\bdealing\b",                    "inflicting"),
    (r"\bon a hit\b",                   "upon striking"),
    (r"\bcreature you can see\b",       "visible target"),
    (r"\byou hurl\b",                   "you launch"),
    (r"\bwithin range\b",               "in range"),
    (r"\bbright streak\b",              "luminous bolt"),
    (r"\byou conjure\b",                "you summon"),
    (r"\bsaving throw\b",               "resistance check"),
    (r"\btakes \d+d\d+ (\w+) damage\b", r"suffers \1 damage"),
]

def augment_desc(desc):
    for pat, rep in SWAPS:
        if re.search(pat, desc, re.I):
            return re.sub(pat, rep, desc, count=1, flags=re.I)
    return desc

In [ ]:
import numpy as np, pandas as pd
np.random.seed(42)

# ── near-duplicate filter ─────────────────────────────────────────────────
# Spells whose descriptions share >85% word tokens get a2d/na2d deduplicated.
# n2d stays (spell names are distinct); only description-gen tasks are affected.
def token_jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / len(sa | sb) if sa | sb else 0.0

seen_descs, dedup_ids = [], set()
for idx, row in df.iterrows():
    d = str(row["desc"])
    if any(token_jaccard(d, s) > 0.85 for s in seen_descs):
        dedup_ids.add(idx)
    else:
        seen_descs.append(d)
print(f"near-duplicate descriptions filtered from a2d/na2d: {len(dedup_ids)}")

# ── example builder ───────────────────────────────────────────────────────
examples = []

for idx, row in df.iterrows():
    name   = str(row["name"])
    desc   = str(row["desc"])
    full   = build_attr(row)
    aug    = augment_desc(desc)
    stype  = infer_type(row)
    is_dup = idx in dedup_ids

    # all tasks except deduplicated description-gen
    if not is_dup:
        examples.append({"input": f"generate description: {full}",  "target": desc, "task": "a2d"})
    examples.append(    {"input": f"generate name: {desc}",         "target": name, "task": "d2n"})
    examples.append(    {"input": f"describe spell: {name}",        "target": desc, "task": "n2d"})
    if not is_dup:
        examples.append({"input": f"describe spell: {name} | school: {row['school']} | level: {row['level']}",
                         "target": desc, "task": "na2d"})
    examples.append(    {"input": f"name this spell: {full}",       "target": name, "task": "a2n"})

    if aug != desc:
        examples.append({"input": f"generate name: {aug}", "target": name, "task": "d2n_aug"})

    if str(row.get("higher_levels", "")).strip():
        examples.append({"input": f"upcast: {name} | {full}", "target": str(row["higher_levels"]), "task": "upcast"})

    if row["dc_type"]:
        examples.append({"input": f"saving throw: {desc}",    "target": row["dc_type"],     "task": "pred_dc"})
    if row["school"]:
        examples.append({"input": f"school of magic: {desc}", "target": row["school"],      "task": "pred_school"})
    if row["damage_type"]:
        examples.append({"input": f"damage type: {desc}",     "target": row["damage_type"], "task": "pred_dmg"})

    # oversample non-aoe description tasks — counters sphere-template dominance
    if stype != "aoe" and not is_dup:
        examples.append({"input": f"generate description: {full}",  "target": desc, "task": "a2d"})
        examples.append({"input": f"describe spell: {name} | school: {row['school']} | level: {row['level']}",
                         "target": desc, "task": "na2d"})

ex_df   = pd.DataFrame(examples)
aug_cap = ex_df[ex_df["task"] == "d2n_aug"].sample(min(500, (ex_df["task"]=="d2n_aug").sum()), random_state=42)
ex_df   = pd.concat([ex_df[ex_df["task"] != "d2n_aug"], aug_cap]).reset_index(drop=True)

df["spell_type"] = df.apply(infer_type, axis=1)
print(f"\ntotal examples : {len(ex_df)}")
print("\nspell type distribution:")
print(df["spell_type"].value_counts().to_string())
print("\nexamples by task:")
print(ex_df["task"].value_counts().to_string())

In [ ]:
import gc

_used  = torch.cuda.memory_allocated() / 1e9
_total = torch.cuda.get_device_properties(0).total_memory / 1e9
_free  = _total - torch.cuda.memory_reserved() / 1e9
print(f"GPU: {_used:.2f} GB allocated | {_free:.2f} GB free | {_total:.2f} GB total")

# The probe runs before optimizer is created.
# AdamW stores 2× model size in moment buffers (~1.16 GB for t5-base).
# Use 0.70 safety factor — training now uses memory-efficient loss so no large float32 cast.

def find_max_batch_size(model, tokenizer, device, min_bs=4, max_bs=512):
    dummy_in  = "describe spell: Fireball"
    dummy_out = "A bright streak of light flashes to a point and explodes in a roar of flame."

    def probe(bs):
        torch.cuda.empty_cache(); gc.collect()
        try:
            enc = tokenizer(
                [dummy_in] * bs,
                max_length=128, truncation=True,
                padding="max_length", return_tensors="pt",
            ).to(device)
            tgt = tokenizer(
                [dummy_out] * bs,
                max_length=256, truncation=True,
                padding="max_length", return_tensors="pt",
            )
            labels = tgt["input_ids"].clone().to(device)
            labels[labels == tokenizer.pad_token_id] = -100

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**enc, labels=labels)
                # memory-efficient label smoothing — stays in bfloat16, no float32 cast
                lse  = torch.logsumexp(outputs.logits, dim=-1)
                mean = outputs.logits.mean(dim=-1)
                mask = labels != -100
                smooth_loss = (lse - mean)[mask].mean()
                loss = 0.9 * outputs.loss + 0.1 * smooth_loss

            loss.backward()
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache(); gc.collect()
            print(f"  bs={bs:4d}  ✓")
            return True
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print(f"  bs={bs:4d}  ✗ OOM")
            return False

    lo, hi = min_bs, min_bs
    while hi <= max_bs and probe(hi):
        lo = hi
        hi = min(hi * 2, max_bs + 1)

    if lo == min_bs and not probe(min_bs):
        print("Even min_bs OOMs — free memory first")
        return min_bs

    best = lo
    low, high = lo + 1, hi - 1
    while low <= high:
        mid = (low + high) // 2
        if probe(mid):
            best = mid; low  = mid + 1
        else:
            high = mid - 1

    torch.cuda.empty_cache(); gc.collect()
    return best


print("probing max batch size…")
_max_bs = find_max_batch_size(model, tokenizer, device, min_bs=4, max_bs=512)

TRAIN_BATCH_SIZE = max(4, int(_max_bs * 0.70))
VAL_BATCH_SIZE   = _max_bs

print(f"\nmax that fits : {_max_bs}")
print(f"train batch   : {TRAIN_BATCH_SIZE}")
print(f"val   batch   : {VAL_BATCH_SIZE}")

In [17]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

train_ex, val_ex = train_test_split(
    ex_df.to_dict("records"),
    test_size=0.1,
    random_state=42,
    stratify=ex_df["task"],
)

train_dataset = SpellDataset(train_ex)
val_dataset   = SpellDataset(val_ex)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print(f"train : {len(train_ex)} examples  |  {len(train_loader)} batches")
print(f"val   : {len(val_ex)}  examples  |  {len(val_loader)} batches")

train : 8397 examples  |  168 batches
val   : 933  examples  |  16 batches


In [ ]:
import math, random
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import get_cosine_schedule_with_warmup
import evaluate as hf_evaluate

EPOCHS    = 25
LR        = 2e-4
SAVE_DIR  = "/content/drive/MyDrive/t5_spells_best"
PATIENCE  = 4
os.makedirs(SAVE_DIR, exist_ok=True)

rouge_metric = hf_evaluate.load("rouge")

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01,
    fused=torch.cuda.is_available(),
)

total_steps = len(train_loader) * EPOCHS
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps,
)

start_epoch        = 0
best_val_loss      = float("inf")
best_rouge_l       = 0.0
patience_count     = 0
train_history      = []
val_history        = []
perplexity_history = []
rouge_history      = []

print(f"ready — {MODEL_NAME} | lr={LR} | epochs={EPOCHS} | patience={PATIENCE}")

In [ ]:
import os, math, random
import torch.nn.functional as F
from tqdm.auto import tqdm

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

SMOOTH       = 0.1
ROUGE_EVAL_N = 50   # how many val examples to score with ROUGE-L each epoch

# pool of description-generating val examples for ROUGE evaluation
_rouge_pool = [e for e in val_ex if e["task"] in ("a2d", "na2d", "n2d")]

PROBE_PROMPTS = [
    ("describe spell: Fireball",                                                    "n2d"),
    ("describe spell: Misty Step",                                                  "n2d"),
    ("generate description: level: 3 | type: aoe | school: Necromancy | damage: Poison | duration: 1 minute",
                                                                                    "a2d aoe"),
    ("generate description: level: 2 | type: teleport | school: Conjuration | range: Self | duration: Instantaneous",
                                                                                    "a2d teleport"),
    ("generate name: Spectral chains restrain enemies and drain their life force.", "d2n"),
]

def label_smoothed_loss(outputs, labels):
    lse  = torch.logsumexp(outputs.logits, dim=-1)
    mean = outputs.logits.mean(dim=-1)
    mask = labels != -100
    smooth_component = (lse - mean)[mask].mean()
    return (1 - SMOOTH) * outputs.loss + SMOOTH * smooth_component

def sample_generate(prompt, max_new_tokens=150):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        ids = model.generate(
            **enc,
            max_new_tokens       = max_new_tokens,
            num_beams            = 4,
            num_beam_groups      = 4,
            diversity_penalty    = 1.0,
            no_repeat_ngram_size = 3,
            repetition_penalty   = 1.3,
            length_penalty       = 1.2,
            early_stopping       = True,
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True)

def unwrap(m):
    return m._orig_mod if hasattr(m, "_orig_mod") else m


for epoch in range(start_epoch, EPOCHS):
    # ── train ──────────────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0.0
    train_bar = tqdm(train_loader, desc=f"Ep {epoch+1:02d}/{EPOCHS} [train]",
                     leave=False, dynamic_ncols=True)

    for batch in train_bar:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss    = label_smoothed_loss(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    train_history.append(avg_train_loss)

    # ── val loss ───────────────────────────────────────────────────────────
    model.eval()
    total_val_loss = 0.0
    val_bar = tqdm(val_loader, desc=f"Ep {epoch+1:02d}/{EPOCHS} [val]  ",
                   leave=False, dynamic_ncols=True)

    with torch.no_grad():
        for batch in val_bar:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs   = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                step_loss = label_smoothed_loss(outputs, labels)
            total_val_loss += step_loss.item()
            val_bar.set_postfix(loss=f"{step_loss.item():.4f}")

    avg_val_loss = total_val_loss / len(val_loader)
    perplexity   = math.exp(avg_val_loss)
    val_history.append(avg_val_loss)
    perplexity_history.append(perplexity)

    # ── ROUGE-L on val sample ──────────────────────────────────────────────
    rouge_subset = random.sample(_rouge_pool, min(ROUGE_EVAL_N, len(_rouge_pool)))
    preds  = [sample_generate(e["input"]) for e in rouge_subset]
    refs   = [e["target"]                 for e in rouge_subset]
    scores = rouge_metric.compute(predictions=preds, references=refs)
    rouge_l = scores["rougeL"]
    rouge_history.append(rouge_l)

    print(f"\nEpoch {epoch+1}/{EPOCHS}  Train: {avg_train_loss:.4f}  Val: {avg_val_loss:.4f}"
          f"  PPL: {perplexity:.2f}  ROUGE-L: {rouge_l:.4f}")

    # ── mid-training probes ────────────────────────────────────────────────
    print("\n  Sample outputs:")
    for prompt, task in PROBE_PROMPTS:
        out = sample_generate(prompt)
        short = prompt[:55] + ("…" if len(prompt) > 55 else "")
        print(f"  [{task}] {short}")
        print(f"    → {out[:150]}")

    # ── checkpoint on best ROUGE-L ─────────────────────────────────────────
    if rouge_l > best_rouge_l:
        best_rouge_l   = rouge_l
        best_val_loss  = avg_val_loss
        patience_count = 0
        raw = unwrap(model)
        raw.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        torch.save({
            "epoch"             : epoch,
            "optimizer_state"   : optimizer.state_dict(),
            "scheduler_state"   : scheduler.state_dict(),
            "best_rouge_l"      : best_rouge_l,
            "best_val_loss"     : best_val_loss,
            "train_history"     : train_history,
            "val_history"       : val_history,
            "perplexity_history": perplexity_history,
            "rouge_history"     : rouge_history,
        }, os.path.join(SAVE_DIR, "training_state.pt"))
        print(f"  ✓ saved best  ROUGE-L={best_rouge_l:.4f}  val={best_val_loss:.4f}")
    else:
        patience_count += 1
        print(f"  no improvement ({patience_count}/{PATIENCE})")
        if patience_count >= PATIENCE:
            print(f"\n  early stop — no ROUGE-L gain for {PATIENCE} epochs")
            break

    print("─" * 60)
    torch.cuda.empty_cache()

In [ ]:
epochs_range = list(range(1, len(train_history) + 1))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Training Summary", fontsize=14, fontweight="bold")

axes[0].plot(epochs_range, train_history, color="steelblue", linewidth=2, marker="o", markersize=3)
axes[0].set_title("Train Loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, val_history, color="darkorange", linewidth=2, marker="o", markersize=3)
if val_history:
    best_epoch = val_history.index(min(val_history)) + 1
    axes[1].axvline(best_epoch, color="red", linestyle="--", alpha=0.6, label=f"best epoch {best_epoch}")
    axes[1].legend()
axes[1].set_title("Validation Loss"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, perplexity_history, color="seagreen", linewidth=2, marker="o", markersize=3)
axes[2].set_title("Validation Perplexity"); axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Perplexity")
axes[2].grid(True, alpha=0.3)

axes[3].plot(epochs_range, rouge_history, color="mediumpurple", linewidth=2, marker="o", markersize=3)
if rouge_history:
    best_rouge_epoch = rouge_history.index(max(rouge_history)) + 1
    axes[3].axvline(best_rouge_epoch, color="red", linestyle="--", alpha=0.6, label=f"best epoch {best_rouge_epoch}")
    axes[3].legend()
axes[3].set_title("ROUGE-L (val sample)"); axes[3].set_xlabel("Epoch"); axes[3].set_ylabel("ROUGE-L")
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

if val_history:
    print(f"best val loss  : {min(val_history):.4f}  (epoch {best_epoch})")
    print(f"best perplexity: {min(perplexity_history):.2f}")
if rouge_history:
    print(f"best ROUGE-L   : {max(rouge_history):.4f}  (epoch {best_rouge_epoch})")

In [ ]:
def generate(prompt, max_new_tokens=200, num_beams=4,
             temperature=1.0, top_p=0.95, do_sample=False):
    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens         = max_new_tokens,
            num_beams              = num_beams,
            num_beam_groups        = num_beams if not do_sample else 1,
            diversity_penalty      = 1.0       if not do_sample else 0.0,
            no_repeat_ngram_size   = 3,
            repetition_penalty     = 1.3,
            length_penalty         = 1.2,
            early_stopping         = True,
            temperature            = temperature,
            top_p                  = top_p if do_sample else 1.0,
            do_sample              = do_sample,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
probe_prompts = [
    "describe spell: Fireball",
    "describe spell: Misty Step",
    "generate description: level: 3 | type: aoe | school: Necromancy | damage: Poison | duration: 1 minute",
    "generate description: level: 7 | type: single_target | school: Conjuration | range: 500 feet | duration: Instantaneous",
    "generate description: level: 2 | type: teleport | school: Conjuration | range: Self | duration: Instantaneous",
    "generate description: level: 0 | type: single_target | school: Evocation | damage: Lightning | range: 60 feet",
    "generate description: level: 6 | type: concentration | school: Transmutation | duration: 24 hours",
    "generate description: level: 1 | type: utility | school: Divination | duration: 10 minutes | ritual: yes",
    "generate name: A wave of freezing wind blasts outward from you, dealing cold damage to creatures in a cone.",
    "generate name: You summon spectral chains that restrain enemies and drain their life force.",
    "describe spell: Ashen Nova | school: Evocation | level: 5",
]

In [ ]:
for i, prompt in enumerate(probe_prompts):


    print(f"Prompt {i + 1}")

    print(prompt)

    print("Output:\n")

    result = generate(
        prompt,
        max_new_tokens=200
    )

    print(result)

    print("\n")